In [ ]:
from openai import OpenAI
from itertools import combinations
import json
openai_api_key = os.getenv("OPENAI_API_KEY", "")
# Initialize OpenAI client with OpenRouter API
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openai_api_key,
)

# Function to load JSON data from file
def load_json_data(filename):
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            content = file.read().strip()
            # 调试打印
            print(f"File content length: {len(content)}")
            print(f"First 100 chars: {content[:100]}")
            
            try:
                # 尝试直接解析完整的JSON
                outer_array = json.loads(content)
                # 如果数据已经是字典列表，直接返回
                if isinstance(outer_array[0], dict):
                    return outer_array
                # 如果是字符串列表，需要进一步解析
                return [json.loads(item) for item in outer_array]
            except json.JSONDecodeError as e:
                print(f"JSON parsing error: {str(e)}")
                # 尝试修复可能的JSON格式问题
                content = content.replace('\\n', ' ').replace('\\r', ' ')
                # 重试解析
                outer_array = json.loads(content)
                return [json.loads(item) for item in outer_array]
    except Exception as e:
        print(f"Error loading file: {str(e)}")
        raise

# Function to count topics in Domain
def count_domain_topics(domain_str):
    return len([t.strip() for t in domain_str.split(",") if t.strip()])

# Function to get LLM similarity judgment for two items
def get_item_similarity_score(item1, item2, domain1, domain2):
    prompt = f"""
    You are an expert judge evaluating the semantic similarity between two tagged items from different rounds of a text analysis task. 
    Both rounds have TaskType "Joint" and QueryType "Cardinality". Assess the similarity based on their semantic meaning and relevance to their respective domains.

    Item 1: {item1}
    Domain 1: {domain1}

    Item 2: {item2}
    Domain 2: {domain2}

    Provide a similarity score between 0 and 1 (where 1 is identical and 0 is completely dissimilar). Respond with only the score (e.g., 0.85).
    """
    response = client.chat.completions.create(
                model="gpt-4.1",
                messages=[
                    {"role": "user", "content": prompt}
                ],
                max_tokens=16000
            )
    return float(response.choices[0].message.content)
def get_group_similarity_score(item_list, domain_list=None):
    prompt = f"""
    You are an expert reviewer. Below are the results of 100 rounds of experiments on the same task. Please comprehensively judge the similarity of these results (0-1 points, 1 is completely consistent, 0 is completely different), and only reply with the score.
    Each result and its field are as follows:
    """
    for idx, (item) in enumerate(zip(item_list), 1):
        prompt += f"\n结果{idx}: {item}"
    prompt += "\nPlease give the overall similarity score of these results (0-1), and reply with numbers only."
    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10000
    )
    return float(response.choices[0].message.content)
# Process data
filename = r"D:/BaiduNetdiskDownload/TADA/TADA/InightPilot.TextAnalysis/TextAnalysis.Test/Output/Stability/Tagging-Output/gpt-4.1/Amazon_100/AP+TbS/Amazon_100_1.json"
# filename="D:/Documents/MSIntern/TextAnalysis/TextAnalysis.Test/Output/Stability/Tagging-Output/gpt-4.1/Amazon_100/TbS/Amazon_100_0.json"

# 加载数据并打印调试信息
data = load_json_data(filename)
print(f"Loaded {len(data)} rounds of data")
print(f"First round structure: {data[0].keys()}")

n_rounds = len(data)
max_items = max(len(round_data["Results"]["Items"]) for round_data in data)  # Assume all have 100 items

# Calculate pairwise combinations and item-wise similarities
# pairwise_similarities = {}
# for i, j in combinations(range(n_rounds), 2):
#     round1, round2 = data[i], data[j]
#     domain1, domain2 = round1["Domain"], round2["Domain"]
#     item_similarities = []
    
#     # Compare items up to the maximum number of items (e.g., 100)
#     for item_idx in range(1, max_items + 1):
#         item1 = round1["Results"]["Items"].get(str(item_idx), "N/A")
#         item2 = round2["Results"]["Items"].get(str(item_idx), "N/A")
#         similarity = get_item_similarity_score(item1, item2, domain1, domain2)
#         item_similarities.append(similarity)
    
#     avg_item_similarity = sum(item_similarities) / len(item_similarities) if item_similarities else 0
#     pairwise_similarities[(i+1, j+1)] = avg_item_similarity
#     print(f"Average Item Similarity between Round {i+1} and Round {j+1}: {avg_item_similarity}")

# # Calculate overall average similarity
# total_similarity = sum(pairwise_similarities.values())
# n_pairs = len(pairwise_similarities)
# overall_avg_similarity = total_similarity / n_pairs if n_pairs else 0
# print(f"Overall Average Similarity: {overall_avg_similarity}")

# # Count topics in each round's domain
# for i, round_data in enumerate(data, 1):
#     topic_count = count_domain_topics(round_data["Domain"])
#     print(f"Round {i} Domain Topic Count: {topic_count}")


##一起给模型
item_similarities = []
for item_idx in range(1, max_items + 1):
    items = [round_data["Results"]["Items"].get(str(item_idx), "N/A") for round_data in data]
    domains = [round_data["Domain"] for round_data in data]
    sim = get_group_similarity_score(items, domains)
    item_similarities.append(sim)
    print(f"Item {item_idx} group similarity: {sim}")

overall_avg = sum(item_similarities) / len(item_similarities) if item_similarities else 0
print(f"Overall average group similarity: {overall_avg}")
import os

# 1. 收集输出内容
output_lines = []
for idx, sim in enumerate(item_similarities, 1):
    output_lines.append(f"Item {idx} group similarity: {sim}")
output_lines.append(f"Overall average group similarity: {overall_avg}")

# 2. 生成txt文件名（同目录，仅后缀变为.txt）
base, _ = os.path.splitext(filename)
txt_filename = base + ".txt"

# 3. 写入txt文件
with open(txt_filename, 'w', encoding='utf-8') as f:
    for line in output_lines:
        f.write(line + '\\n')

print(f"Results saved to {txt_filename}")

In [ ]:
import glob
import os

# 假设文件都在同一个目录下
json_dir = "Output/Stability/Tagging-Output/gpt-4.1/Amazon_100/AP+TbS/"
json_files = ["Output/Stability/Tagging-Output/gpt-4.1/Amazon_100/AP/Amazon_100_0.json","Output/Stability/Tagging-Output/gpt-4.1/Amazon_100/AP/Amazon_100_1.json","Output/Stability/Tagging-Output/gpt-4.1/Amazon_100/AP/Amazon_100_2.json","Output/Stability/Tagging-Output/gpt-4.1/Amazon_100/TbS/Amazon_100_0.json","Output/Stability/Tagging-Output/gpt-4.1/Amazon_100/TbS/Amazon_100_1.json","Output/Stability/Tagging-Output/gpt-4.1/Amazon_100/TbS/Amazon_100_2.json","Output/Stability/Tagging-Output/gpt-4.1/Amazon_100/none/Amazon_100_0.json","Output/Stability/Tagging-Output/gpt-4.1/Amazon_100/none/Amazon_100_1.json","Output/Stability/Tagging-Output/gpt-4.1/Amazon_100/none/Amazon_100_2.json"]

def process_file(filename):
    data = load_json_data(filename)
    n_rounds = len(data)
    max_items = max(len(round_data["Results"]["Items"]) for round_data in data)
    item_similarities = []
    output_lines = []
    for item_idx in range(1, max_items + 1):
        items = [round_data["Results"]["Items"].get(str(item_idx), "N/A") for round_data in data]
        domains = [round_data["Domain"] for round_data in data]
        sim = get_group_similarity_score(items, domains)
        item_similarities.append(sim)
        line = f"Item {item_idx} group similarity: {sim}"
        print(line)
        output_lines.append(line)
    overall_avg = sum(item_similarities) / len(item_similarities) if item_similarities else 0
    avg_line = f"Overall average group similarity: {overall_avg}"
    print(avg_line)
    output_lines.append(avg_line)
    # 输出到txt文件
    base, _ = os.path.splitext(filename)
    txt_filename = base + ".txt"
    with open(txt_filename, 'w', encoding='utf-8') as f:
        for line in output_lines:
            f.write(line + '\\n')
    print(f"Results saved to {txt_filename}")


def load_json_data(filename):
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            content = file.read().strip()
            if not content:
                print(f"Warning: {filename} is empty!")
                return []
            outer_array = json.loads(content)
            # 如果第一个元素是字符串（即字符串化的json），需要再解析一遍
            if isinstance(outer_array[0], str):
                return [json.loads(item) for item in outer_array]
            # 否则直接返回
            return outer_array
    except Exception as e:
        print(f"Error loading file {filename}: {e}")
        return []
        
def process_file(filename):
    data = load_json_data(filename)
    n_rounds = len(data)
    max_items = max(len(round_data["Results"]["Items"]) for round_data in data)
    item_similarities = []
    output_lines = []
    for item_idx in range(1, max_items + 1):
        items = [round_data["Results"]["Items"].get(str(item_idx), "N/A") for round_data in data]
        sim = get_group_similarity_score(items)
        item_similarities.append(sim)
        line = f"Item {item_idx} group similarity: {sim}"
        print(line)
        output_lines.append(line)
    overall_avg = sum(item_similarities) / len(item_similarities) if item_similarities else 0
    avg_line = f"Overall average group similarity: {overall_avg}"
    print(avg_line)
    output_lines.append(avg_line)
    # 输出到txt文件
    base, _ = os.path.splitext(filename)
    txt_filename = base + ".txt"
    with open(txt_filename, 'w', encoding='utf-8') as f:
        for line in output_lines:
            f.write(line + '\\n')
    print(f"Results saved to {txt_filename}")
with open(filename, 'r', encoding='utf-8') as file:
    content = file.read().strip()
    print(f"First 200 chars: {repr(content[:200])}")
    print(f"Last 200 chars: {repr(content[-200:])}")
for filename in json_files:
    print(f"Processing {filename} ...")
    process_file(filename)